In [ ]:
%%capture
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por poppler-utils
!pip install -q pandas openpyxl pypdf reportlab pdf2image pytesseract qrcode
print("Instalação concluída.")

In [ ]:
import io
import re
import csv
from pathlib import Path
from datetime import datetime

import pandas as pd
from pdf2image import convert_from_path
from pypdf import PdfReader, PdfWriter
import pytesseract
from reportlab.pdfgen import canvas
from reportlab.lib.colors import Color
from zoneinfo import ZoneInfo

FUSO_BRASILIA = ZoneInfo("America/Sao_Paulo")

# --------------------------------------------------------------------------
# CONFIGURAÇÃO — ajuste aqui se precisar (colunas da planilha, aparência etc.)
# --------------------------------------------------------------------------
PASTA_BASE = Path("/content/carimbo_tool")
PASTA_ENTRADA = PASTA_BASE / "entrada"
PASTA_SAIDA = PASTA_BASE / "saida"
PASTA_LOGS = PASTA_BASE / "logs"
PASTA_PLANILHA = PASTA_BASE / "planilha"
CAMINHO_PLANILHA = PASTA_PLANILHA / "cadastro.xlsx"

for p in [PASTA_ENTRADA, PASTA_SAIDA, PASTA_LOGS, PASTA_PLANILHA]:
    p.mkdir(parents=True, exist_ok=True)

# Colunas usadas pela LÓGICA (busca na planilha). Precisam bater com o
# cabeçalho exato da sua planilha, mesmo que não apareçam no carimbo.
COL_CNPJ = "CNPJ"
COL_FORNECEDOR = "FORNECEDOR"

# Colunas que aparecem IMPRESSAS no carimbo (só exibição).


COD_CODIGO_FORN = "CODIGO FORN"
COD_DO_PRODUTO = "COD PRODUTO"
COD_CONTA_CONTABIL = "CONTA CONTÁBIL"
COD_CENTRO_DE_CUSTO = "CENTRO DE CUSTO"
COD_NATUREZA = "NATUREZA"
COD_TES = "TES"
COD_SERVICO = "COD. SERVIÇO"
COD_PEDIDO = "PEDIDO"
COD_FORNECEDOR = "FORNECEDOR"


CAMPOS_NO_CARIMBO = [
    COD_CODIGO_FORN,COD_DO_PRODUTO,COD_CONTA_CONTABIL,COD_CENTRO_DE_CUSTO,COD_NATUREZA,COD_TES,COD_SERVICO,COD_PEDIDO,COD_FORNECEDOR
]

CARIMBO_LARGURA = 260
CARIMBO_MARGEM_X = 20
CARIMBO_MARGEM_Y = 20
CARIMBO_PAGINA = 0          # 0 = só a primeira página. Use "todas" para carimbar todas.
CARIMBO_COR_BORDA = (0.75, 0.15, 0.15)
CARIMBO_TAMANHO_FONTE = 7.5
INCLUIR_DATA_HORA = True

# Altura calculada automaticamente: cabeçalho + 1 linha por campo (incluindo a data,
# que é uma linha normal da lista) + respiro. Nunca estoura, ajusta sozinho.
_ALTURA_CABECALHO = 20
_ALTURA_POR_LINHA = CARIMBO_TAMANHO_FONTE + 3
_QTD_LINHAS = len(CAMPOS_NO_CARIMBO) + (1 if INCLUIR_DATA_HORA else 0)
_RESPIRO = 10

CARIMBO_ALTURA = int(_ALTURA_CABECALHO + _QTD_LINHAS * _ALTURA_POR_LINHA + _RESPIRO)

DPI_OCR = 300
COLUNAS_LOG = ["arquivo", "cnpj_detectado", "status", "empresa_encontrada", "protocolo", "detalhe"]

# --------------------------------------------------------------------------
# CNPJ: encontrar e VALIDAR (evita carimbar com dado errado por erro de OCR)
# --------------------------------------------------------------------------
REGEX_CNPJ = re.compile(r"\d{2}\.?\d{3}\.?\d{3}\/?\d{4}-?\d{2}")


def _somente_digitos(texto):
    return re.sub(r"\D", "", texto)


def cnpj_valido(cnpj):
    cnpj = _somente_digitos(cnpj)
    if len(cnpj) != 14 or cnpj == cnpj[0] * 14:
        return False

    def calcula_dv(parcial, pesos):
        soma = sum(int(d) * p for d, p in zip(parcial, pesos))
        resto = soma % 11
        return "0" if resto < 2 else str(11 - resto)

    p1 = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    p2 = [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    dv1 = calcula_dv(cnpj[:12], p1)
    dv2 = calcula_dv(cnpj[:12] + dv1, p2)
    return cnpj[-2:] == dv1 + dv2


def formatar_cnpj(cnpj):
    d = _somente_digitos(cnpj)
    if len(d) != 14:
        return cnpj
    return f"{d[0:2]}.{d[2:5]}.{d[5:8]}/{d[8:12]}-{d[12:14]}"


def extrair_cnpjs(texto):
    candidatos = REGEX_CNPJ.findall(texto)
    validos = []
    for c in candidatos:
        d = _somente_digitos(c)
        if cnpj_valido(d) and d not in validos:
            validos.append(d)
    return validos


# --------------------------------------------------------------------------
# Extração: texto nativo do PDF primeiro; OCR só se necessário (escaneado)
# --------------------------------------------------------------------------
def _texto_nativo(caminho_pdf):
    try:
        reader = PdfReader(caminho_pdf)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    except Exception:
        return ""


def _texto_via_ocr(caminho_pdf, max_paginas=2):
    imagens = convert_from_path(caminho_pdf, dpi=DPI_OCR, last_page=max_paginas)
    texto = ""
    for imagem in imagens:
        texto += pytesseract.image_to_string(imagem, lang="por") + "\n"
    return texto


def extrair_cnpj_do_pdf(caminho_pdf):
    texto = _texto_nativo(caminho_pdf)
    cnpjs = extrair_cnpjs(texto)
    if cnpjs:
        return cnpjs[0], "texto_nativo", texto
    texto_ocr = _texto_via_ocr(caminho_pdf)
    cnpjs_ocr = extrair_cnpjs(texto_ocr)
    texto_completo = texto + "\n" + texto_ocr
    if cnpjs_ocr:
        return cnpjs_ocr[0], "ocr", texto_completo
    return None, "nao_encontrado", texto_completo


# --------------------------------------------------------------------------
# Planilha: carregar e buscar por CNPJ (ou, se falhar, por nome do fornecedor)
# --------------------------------------------------------------------------
def carregar_planilha(caminho=None):
    caminho = caminho or CAMINHO_PLANILHA
    df = pd.read_excel(caminho, dtype=str)
    if COL_CNPJ not in df.columns:
        raise ValueError(f"A planilha precisa ter uma coluna chamada '{COL_CNPJ}'. Colunas encontradas: {list(df.columns)}")
    df["_cnpj_normalizado"] = df[COL_CNPJ].apply(lambda v: _somente_digitos(str(v)) if pd.notna(v) else "")
    return df


def buscar_por_cnpj(df, cnpj):
    cnpj = _somente_digitos(cnpj)
    resultado = df[df["_cnpj_normalizado"] == cnpj]
    if resultado.empty:
        return None
    return resultado.iloc[0].to_dict()


def _normalizar(txt):
    import unicodedata
    txt = str(txt).upper()
    return unicodedata.normalize("NFKD", txt).encode("ASCII", "ignore").decode("ASCII")


def buscar_por_nome(df, texto_documento):
    # Fallback: procura o nome da empresa (coluna Empresa) dentro do texto do PDF.
    texto_norm = _normalizar(texto_documento)
    for _, row in df.iterrows():
        nome_empresa = row.get(COL_EMPRESA, "")
        if not nome_empresa:
            continue
        nome_norm = _normalizar(nome_empresa)
        if nome_norm and nome_norm in texto_norm:
            return row.to_dict()
    return None


# --------------------------------------------------------------------------
# Carimbo: desenha uma camada PDF separada e sobrepõe no documento original
# --------------------------------------------------------------------------
def _gerar_protocolo(cnpj):
    return f"{cnpj[:8]}-{datetime.now(FUSO_BRASILIA).strftime('%Y%m%d%H%M%S')}"


def _desenhar_overlay(dados, largura_pagina, altura_pagina):
    buf = io.BytesIO()
    c = canvas.Canvas(buf, pagesize=(largura_pagina, altura_pagina))
    x0 = largura_pagina - CARIMBO_LARGURA - CARIMBO_MARGEM_X
    y0 = CARIMBO_MARGEM_Y
    cor_borda = Color(*CARIMBO_COR_BORDA)

    c.setFillColor(cor_borda)
    c.setFont("Helvetica-Bold", CARIMBO_TAMANHO_FONTE + 1)
    c.drawString(x0 + 8, y0 + CARIMBO_ALTURA - 14, "CARIMBO DIGITAL")

    linhas = [(campo, str(dados.get(campo, "") or "")) for campo in CAMPOS_NO_CARIMBO]
    if INCLUIR_DATA_HORA:
        agora_brasilia = datetime.now(FUSO_BRASILIA)
        linhas.append(("Data", agora_brasilia.strftime("%d/%m/%Y %H:%M")))

    c.setFont("Helvetica", CARIMBO_TAMANHO_FONTE)
    c.setStrokeColor(cor_borda)
    c.setLineWidth(0.5)
    linha_y = y0 + CARIMBO_ALTURA - 28
    largura_texto = CARIMBO_LARGURA - 8
    for campo, valor in linhas:
        max_chars = int(largura_texto / (CARIMBO_TAMANHO_FONTE * 0.5))
        if len(valor) > max_chars:
            valor = valor[:max_chars - 1] + "…"
        texto_linha = f"{campo}: {valor}"
        c.drawString(x0 + 8, linha_y, texto_linha)
        # Sublinhado: uma linha fina logo abaixo do texto, do mesmo comprimento dele
        largura_linha = c.stringWidth(texto_linha, "Helvetica", CARIMBO_TAMANHO_FONTE)
        c.line(x0 + 8, linha_y - 2, x0 + 8 + largura_linha, linha_y - 2)
        linha_y -= CARIMBO_TAMANHO_FONTE + 3
        if linha_y < y0 + 6:
            break

    c.save()
    buf.seek(0)
    return buf

def aplicar_carimbo(caminho_pdf_entrada, caminho_pdf_saida, dados_empresa, cnpj):
    protocolo = _gerar_protocolo(cnpj)
    reader = PdfReader(caminho_pdf_entrada)
    writer = PdfWriter()
    paginas_alvo = range(len(reader.pages)) if CARIMBO_PAGINA == "todas" else [CARIMBO_PAGINA]

    for i, page in enumerate(reader.pages):
        if i in paginas_alvo:
            largura = float(page.mediabox.width)
            altura = float(page.mediabox.height)
            overlay_buf = _desenhar_overlay(dados_empresa, largura, altura)
            overlay_page = PdfReader(overlay_buf).pages[0]
            page.merge_page(overlay_page)
        writer.add_page(page)

    with open(caminho_pdf_saida, "wb") as f:
        writer.write(f)
    return protocolo


print("Ferramenta carregada. Siga para a próxima célula.")

Ferramenta carregada. Siga para a próxima célula.


In [ ]:
from google.colab import files
import shutil

print("Selecione o arquivo .xlsx da planilha de cadastro:")
enviado = files.upload()
nome_arquivo = list(enviado.keys())[0]
shutil.move(nome_arquivo, CAMINHO_PLANILHA)
print(f"\nPlanilha salva em: {CAMINHO_PLANILHA}")

df_preview = pd.read_excel(CAMINHO_PLANILHA)
df_preview.head()

Selecione o arquivo .xlsx da planilha de cadastro:


Saving Cópia de CADASTRO FORNEC - SETORES (003).xlsx to Cópia de CADASTRO FORNEC - SETORES (003) (1).xlsx

Planilha salva em: /content/carimbo_tool/planilha/cadastro.xlsx


,REGIME TRIB OPTANTE,RESPONSAVÉL PELO PEDIDO,CODIGO FORN,FORNECEDOR,CNPJ,COD PRODUTO,CONTA CONTABIL,CENTRO DE CUSTO,NATUREZA,TES,...,Tipo de Serviço,Municipio,Transmite,Retem ISS,Retem PIS,Retem Cofins,Retem CSLL,Retem IRF,Retem INSS,OBSERVAÇÃO
0,Não,DAIANE - FAZ PEDIDO,000443,10 Tabelião de Protesto de letrase titulos da ...,59.950.535/0001-76,DDD057,4400115,1101,104015,132.0,...,"Serviços de registros públicos, cartorários e ...",São Paulo,homologado,Não,Não,Não,Não,Não,Não,NaN
1,SIM,Compras faz a solicitação de compras,003453,3 G Comercio & locação LTDA,15.605.518/0001-26,DDD060,4290112,1302,105007,131.0,...,Organização de festas e recepções; bufê (excet...,São Paulo,homologado,Não,Não,Não,Não,Não,Não,NaN
2,Não,DAIANE - FAZ PEDIDO,000439,3 tabelião de protesto de letras e titulo,54.198.908/0001-80,DDD057,4270123,1204,104015,132.0,...,"Serviços de registros públicos, cartorários e ...",São Paulo,homologado,Não,Não,Não,Não,Não,Não,NaN
3,Não,DAIANE - FAZ PEDIDO,000581,4 Tabelionato de protesto de titulos,59.945.840/0001-70,DDD057,4270123,1204,104015,132.0,...,"Serviços de registros públicos, cartorários e ...",São Paulo,homologado,Não,Não,Não,Não,Não,Não,NaN
4,Não,DAIANE - FAZ PEDIDO,004532,6 Tabelião de Protesto de Letras e Titulo,52.751.994/0001-81,DDD057,4270123,NaN,104015,132.0,...,"Serviços de registros públicos, cartorários e ...",São Paulo,homologado,Não,Não,Não,Não,Não,Não,NaN


In [ ]:
print("Selecione um ou mais PDFs:")
enviados = files.upload()

# Limpa a pasta de entrada antes de receber os novos arquivos, para que PDFs
# de uma execução anterior não fiquem acumulados/armazenados junto dos novos.
for pdf_antigo in PASTA_ENTRADA.glob("*.pdf"):
    pdf_antigo.unlink()

for nome in enviados.keys():
    if nome.lower().endswith(".pdf"):
        shutil.move(nome, PASTA_ENTRADA / nome)

pdfs = sorted(PASTA_ENTRADA.glob("*.pdf"))
print(f"\n{len(pdfs)} PDF(s) prontos para processar:")
for p in pdfs:
    print(" -", p.name)

Selecione um ou mais PDFs:


Saving NFe_35188001230262573000154000000000018626081775996623_30262573000154_19082026020435 (003).pdf to NFe_35188001230262573000154000000000018626081775996623_30262573000154_19082026020435 (003).pdf

1 PDF(s) prontos para processar:
 - NFe_35188001230262573000154000000000018626081775996623_30262573000154_19082026020435 (003).pdf


In [ ]:
df_cadastro = carregar_planilha()
linhas_log = []
pdfs = sorted(PASTA_ENTRADA.glob("*.pdf"))

if not pdfs:
    print("Nenhum PDF encontrado. Volte à célula anterior e envie os arquivos.")

for caminho_pdf in pdfs:
    nome = caminho_pdf.name
    print(f"\nProcessando: {nome}")

    cnpj, metodo, texto_doc = extrair_cnpj_do_pdf(str(caminho_pdf))

    dados = None
    origem_match = ""

    if cnpj:
        print(f"  -> CNPJ detectado ({metodo}): {formatar_cnpj(cnpj)}")
        dados = buscar_por_cnpj(df_cadastro, cnpj)
        if dados:
            origem_match = "CNPJ"

    if not dados:
        dados = buscar_por_nome(df_cadastro, texto_doc)
        if dados:
            origem_match = "nome do fornecedor"
            print(f"  -> CNPJ não bateu, mas encontrado pelo nome do fornecedor: {dados.get(COL_EMPRESA)}")

    if not dados:
        print("  -> Não foi possível identificar a empresa (nem por CNPJ, nem por nome). Pulado.")
        linhas_log.append({"arquivo": nome, "cnpj_detectado": formatar_cnpj(cnpj) if cnpj else "",
                            "status": "NAO_IDENTIFICADO", "empresa_encontrada": "",
                            "protocolo": "", "detalhe": "CNPJ inválido/não cadastrado e nome não encontrado no texto."})
        continue

    caminho_saida = PASTA_SAIDA / f"carimbado_{nome}"
    try:
                protocolo = aplicar_carimbo(str(caminho_pdf), str(caminho_saida), dados, cnpj or dados.get(COL_CNPJ, ""))
    except Exception as e:
        print(f"  -> Erro ao carimbar: {e}")
        linhas_log.append({"arquivo": nome, "cnpj_detectado": formatar_cnpj(cnpj), "status": "ERRO",
                            "empresa_encontrada": dados.get("Empresa", ""), "protocolo": "", "detalhe": str(e)})
        continue

    print(f"  -> Carimbado com sucesso -> {caminho_saida.name}")
    linhas_log.append({"arquivo": nome, "cnpj_detectado": formatar_cnpj(cnpj), "status": "OK",
                        "empresa_encontrada": dados.get("Empresa", ""), "protocolo": protocolo, "detalhe": ""})

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
caminho_log = PASTA_LOGS / f"relatorio_{ts}.csv"
with open(caminho_log, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=COLUNAS_LOG)
    writer.writeheader()
    writer.writerows(linhas_log)

ok = sum(1 for l in linhas_log if l["status"] == "OK")
print(f"\nConcluído: {ok}/{len(linhas_log)} documentos carimbados.")

pd.DataFrame(linhas_log)


Processando: NFe_35188001230262573000154000000000018626081775996623_30262573000154_19082026020435 (003).pdf
  -> CNPJ detectado (texto_nativo): 30.262.573/0001-54
  -> Carimbado com sucesso -> carimbado_NFe_35188001230262573000154000000000018626081775996623_30262573000154_19082026020435 (003).pdf

Concluído: 1/1 documentos carimbados.


,arquivo,cnpj_detectado,status,empresa_encontrada,protocolo,detalhe
0,NFe_351880012302625730001540000000000186260817...,30.262.573/0001-54,OK,,30262573-20260820095144,


In [ ]:
import shutil as _shutil

zip_path = "/content/resultado_carimbos"
_shutil.make_archive(zip_path, "zip", PASTA_SAIDA)

files.download(zip_path + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>